# TTS Dataset Enhancement

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Reads a manifest that already has audio and writes an enhanced copy of every
row as a new row, using [audiosronnx](https://github.com/TigreGotico/audiosronnx),
a pure-onnxruntime audio super-resolution and speech-restoration library (no
torch at runtime). Nothing here deletes or moves the parent audio, and no row
is skipped for a quality reason: this notebook only enhances. A later,
separate notebook measures the result, and another one decides what a trainer
keeps.

The manifest format is [`docs/manifest.md`](../docs/manifest.md). This
notebook is step 3 of the chain described in
[`docs/pipeline.md`](../docs/pipeline.md): its input is the output of donor
synthesis or corpus ingest (steps 2a/2b), and its output feeds the measure
notebook (step 4). It never picks an engine from a language code -- the engine
is an explicit, named argument (`ENHANCE_ENGINE`), documented as a candidate
for evaluation, per [`docs/registry.md`](../docs/registry.md).

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set the manifest path, the engine and every parameter via env vars |
| 1 | Install | Fetch `audiosronnx` and the onnxruntime build named by `ONNX_RUNTIME` |
| 2 | Load manifest | Read the input manifest (or, with `SMOKE=True` and no manifest, build a small one) |
| 3 | Load the enhancer | Resolve onnxruntime providers, load the named `audiosronnx` engine |
| 4 | Enhance every row | Restore each clip, write `<name>.enhanced.jsonl` and its `.meta.json` sidecar |
| 5 | Output summary | Confirm the output manifest holds exactly one row per input row |

### Output structure
```
<manifest_dir>/
  <name>.jsonl                 # input, unchanged
  <name>.enhanced.jsonl        # one new row per input row
  <name>.enhanced.jsonl.meta.json
  enhanced/
    <parent-id>-enh.wav
    ...
```

> **Credits:** Enhancement is [audiosronnx](https://github.com/TigreGotico/audiosronnx),
> a pure-onnxruntime multi-engine library with no torch at runtime. The
> default engine wraps [CallEnhancer](https://huggingface.co/Scicom-intl/CallEnhancer)
> (Scicom-intl, CC-BY-NC-4.0). The SMOKE path uses `novasr` instead (about
> 220 KB), so the smoke run needs no multi-hundred-MB download.

| Field | Value |
|---|---|
| Inputs | `ENH_MANIFEST`: a JSON Lines manifest with `id`, `text`, `lang`, `audio`, `duration_s`, `sample_rate`, `source` on every row (docs/manifest.md). With `SMOKE=True` and no manifest at `ENH_MANIFEST`, the notebook builds one from up to `N_SAMPLES` short edge-tts clips. |
| Outputs | `<name>.enhanced.jsonl` (one new row per input row) and its `.meta.json` sidecar; the enhanced WAVs under `enhanced/`. |
| Runtime | SMOKE (novasr, a couple of demo clips): well under a minute on CPU including the model download. Full run with the default `callenhancer` engine: seconds per clip once its weights are cached, plus the one-time download below. |
| Hardware | CPU only; every engine runs on onnxruntime (`ONNX_RUNTIME`/`ONNX_PROVIDERS`). No GPU is required. |
| Cost | No paid compute or API calls. `novasr` (the SMOKE engine) is about 220 KB; `callenhancer` (the full-run default) downloads close to 3 GB of free public ONNX weights from Hugging Face Hub on first use (2.2 GB feature extractor + 0.7 GB decoder; a smaller int8 feature extractor is available but audibly lossy on this model). |

CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path. The full-run default engine is a multi-gigabyte download and is never exercised by CI; SMOKE's `novasr` engine (about 220 KB) is what CI runs.


In [ ]:
# Parameters
SMOKE = False    # True: ENHANCE_ENGINE defaults to the tiny novasr engine, and a built-in demo
                 # manifest (edge-tts clips) is built when ENH_MANIFEST does not exist
N_SAMPLES = 32   # real manifest: rows are capped to this many; demo: this many clips are built (max 3)


---
## 0 · Global Configuration

All parameters are set as environment variables via `os.environ.setdefault()`.
Override any of them from your shell, a `.env` file, or `%env` magic **before**
running the config cell.

**Edit the values below to match your setup, then run the cell.**

| Env var | Meaning |
|---|---|
| `ENH_MANIFEST` | Path to the input manifest: JSON Lines, [manifest.md](../docs/manifest.md) |
| `ENH_AUDIO_DIR` | Subdirectory, relative to the manifest's own directory, that holds the enhanced WAVs |
| `ENH_FORCE` | `1` re-enhances every row, even one whose output WAV already exists; default `0` (resume-safe) |
| `ENHANCE_ENGINE` | `audiosronnx` `load_sr()` engine alias; empty runs no restoration/bandwidth-extension stage. Default `callenhancer`. `audiosronnx.available_models()` prints every registered engine (both kinds). |
| `DENOISE_ENGINE` | `audiosronnx` `load_denoise()` engine alias; empty (default) runs no denoise stage. When both are set, the denoise stage runs first, and `ENHANCE_ENGINE` sees its output. `audiosronnx.available_denoisers()` prints the rest. At least one of `ENHANCE_ENGINE` / `DENOISE_ENGINE` must be set — Stage 3 refuses to run with neither. |
| `ENHANCE_<OPTION>` / `DENOISE_<OPTION>` | One env var per constructor option of the selected engine on each side, e.g. `ENHANCE_PRECISION`, `DENOISE_MODEL`; empty means the engine's own default. The option tables below list every option every engine takes. Setting one an engine does not accept stops the notebook, naming the engine and the option. |
| `TARGET_SR` | The final audio (48 kHz out of every `audiosronnx` stage) is resampled once to this rate |
| `ENH_CACHE_DIR` | Where `audiosronnx` caches downloaded ONNX weights, for both `ENHANCE_ENGINE` and `DENOISE_ENGINE`; empty uses its own default (`~/.local/share/audiosronnx`). Defaults to `HF_HUB_CACHE` when that is set — never point this at `HF_HOME` |
| `ONNX_PROVIDERS` | onnxruntime execution providers: a comma list, or `auto` (CUDA, ROCm, MIGraphX, CPU, CPU always last) |
| `ONNX_RUNTIME` | Which onnxruntime build the install cell fetches: `cpu`, `cuda`, `rocm` |

In [ ]:
import os


def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return f"./{name}"


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


# ──────────────────────────────────────────────
# INPUT / OUTPUT  (Stage 2, 4)
# ──────────────────────────────────────────────
os.environ.setdefault("ENH_MANIFEST", "manifest.jsonl")  # Input manifest with audio: JSON Lines, docs/manifest.md
os.environ.setdefault("ENH_AUDIO_DIR", "enhanced")        # Subdirectory (relative to the manifest's directory) for the enhanced WAVs
os.environ.setdefault("ENH_FORCE", "0")                   # 1 = re-enhance every row, even one whose output WAV already exists

# ──────────────────────────────────────────────
# ENHANCER / DENOISER  (audiosronnx, Stage 3, 4)
#   load_sr(ENHANCE_ENGINE, **kwargs) runs the restoration / bandwidth-extension
#   stage; load_denoise(DENOISE_ENGINE, **kwargs) runs the denoise stage, BEFORE
#   ENHANCE_ENGINE when both are set. Either may be empty, but not both: Stage 3
#   refuses to run with nothing to do. One env var per constructor option of the
#   SELECTED engine on each side, named ENHANCE_<OPTION> or DENOISE_<OPTION>; an
#   empty value means "use the engine's own default". Setting an option the
#   selected engine does not accept stops the notebook (Stage 3), naming the
#   engine and the option. The next cell documents every option every engine
#   takes; audiosronnx.available_models() / available_denoisers() list the
#   registered aliases.
# ──────────────────────────────────────────────
os.environ.setdefault("ENHANCE_ENGINE", "novasr" if SMOKE else "callenhancer")   # audiosronnx load_sr() engine alias; empty = no restoration stage; SMOKE defaults to the tiny novasr engine
os.environ.setdefault("DENOISE_ENGINE", "")                # audiosronnx load_denoise() engine alias; empty (default) = no denoise stage

os.environ.setdefault("ENHANCE_REVISION",      "")   # str   — every ENHANCE_ENGINE: pin an HF revision instead of the adapter's own default
os.environ.setdefault("ENHANCE_FE_PATH",       "")   # str   — callenhancer, sidon: local feature-extractor ONNX path override
os.environ.setdefault("ENHANCE_DECODER_PATH",  "")   # str   — callenhancer, sidon: local decoder/vocoder ONNX path override
os.environ.setdefault("ENHANCE_CHUNK_SECONDS", "")   # float — callenhancer only
os.environ.setdefault("ENHANCE_PRECISION",     "")   # str   — callenhancer only: "fp32" or "int8"
os.environ.setdefault("ENHANCE_STEPS",         "")   # int   — flowhigh only
os.environ.setdefault("ENHANCE_SEED",          "")   # int   — flowhigh only
os.environ.setdefault("ENHANCE_MERGE",         "")   # bool  — flowhigh only
os.environ.setdefault("ENHANCE_FLOW_PATH",     "")   # str   — flowhigh only
os.environ.setdefault("ENHANCE_VOCODER_PATH",  "")   # str   — flowhigh only
os.environ.setdefault("ENHANCE_DENOISE",       "")   # bool  — lavasr only: its own optional pre-denoise pass, distinct from DENOISE_ENGINE
os.environ.setdefault("ENHANCE_CUTOFF_HZ",     "")   # float — lavasr only
os.environ.setdefault("ENHANCE_CHUNK_SAMPLES", "")   # int   — sidon only

os.environ.setdefault("DENOISE_REVISION",         "")   # str   — every DENOISE_ENGINE: pin an HF revision instead of the adapter's own default
os.environ.setdefault("DENOISE_ONNX_PATH",        "")   # str   — cmgan, frcrn, gtcrn, mossformergan, dpdfnet, metadenoiser, mpsenet
os.environ.setdefault("DENOISE_MASK_ONLY",        "")   # bool  — deepfilternet only
os.environ.setdefault("DENOISE_ENC_PATH",         "")   # str   — deepfilternet only
os.environ.setdefault("DENOISE_ERB_DEC_PATH",     "")   # str   — deepfilternet only
os.environ.setdefault("DENOISE_DF_DEC_PATH",      "")   # str   — deepfilternet only
os.environ.setdefault("DENOISE_MODEL",            "")   # str   — dpdfnet, metadenoiser, mpsenet (each has its own set of choices)
os.environ.setdefault("DENOISE_ATTN_LIMIT_DB",    "")   # float — dpdfnet only
os.environ.setdefault("DENOISE_DITHER",           "")   # float — mossformer2 only
os.environ.setdefault("DENOISE_ANALYSIS_PATH",    "")   # str   — voicefixer only
os.environ.setdefault("DENOISE_VOCODER_PATH",     "")   # str   — voicefixer only
os.environ.setdefault("DENOISE_MEL_FILTERS_PATH", "")   # str   — voicefixer only

os.environ.setdefault("TARGET_SR", "24000")           # the final 48 kHz output is resampled once to this rate

# ──────────────────────────────────────────────
# ONNX RUNTIME  (Stage 1, 3)
# ──────────────────────────────────────────────
os.environ.setdefault("ONNX_PROVIDERS", "auto")   # comma list of onnxruntime providers, or "auto"
os.environ.setdefault("ONNX_RUNTIME", "cpu")      # cpu | cuda | rocm — which onnxruntime build the install cell fetches

# ──────────────────────────────────────────────
# MODEL CACHE
#   audiosronnx downloads its ONNX weights from Hugging Face Hub on first use,
#   through its own cache_dir argument — not the Hub's HF_HOME. Point this at
#   a disk with room; it defaults to HF_HUB_CACHE when that variable is set,
#   and to audiosronnx's own default otherwise.
# ──────────────────────────────────────────────
os.environ.setdefault("ENH_CACHE_DIR", os.environ.get("HF_HUB_CACHE", ""))

_set_enhance_options = {
    k: v for k, v in sorted(os.environ.items())
    if k.startswith("ENHANCE_") and k != "ENHANCE_ENGINE" and v.strip()
}
_set_denoise_options = {
    k: v for k, v in sorted(os.environ.items())
    if k.startswith("DENOISE_") and k != "DENOISE_ENGINE" and v.strip()
}

print("Configuration loaded.")
print(f"  Input manifest  : {_env('ENH_MANIFEST')}")
print(f"  Enhanced audio  : {_env('ENH_AUDIO_DIR')} (relative to the manifest's directory)")
print(f"  Enhance engine  : {_env('ENHANCE_ENGINE') or '(none — no restoration stage)'}  options={_set_enhance_options or '(engine defaults)'}")
print(f"  Denoise engine  : {_env('DENOISE_ENGINE') or '(none — no denoise stage)'}  options={_set_denoise_options or '(engine defaults)'}")
print(f"  Target rate     : {_env('TARGET_SR')} Hz")
print(f"  ONNX providers  : {_env('ONNX_PROVIDERS')}  (onnxruntime build: {_env('ONNX_RUNTIME')})")
print(f"  Cache dir       : {_env('ENH_CACHE_DIR') or '(audiosronnx default)'}")
print(f"  Force re-enhance: {_flag('ENH_FORCE')}")

---
## 0a · Enhancer and Denoiser Engine Options

Every `audiosronnx` engine's constructor is read through
`inspect.signature` before use (Stage 3): only the options in the table for
the selected `ENHANCE_ENGINE` or `DENOISE_ENGINE` are read from the
environment, and setting an option an engine does not accept stops the
notebook, naming the engine and the option. `providers` and `cache_dir` are
not in these tables — every engine takes them, and they come from
`ONNX_PROVIDERS` and `ENH_CACHE_DIR` (above) for both `ENHANCE_ENGINE` and
`DENOISE_ENGINE`, resolved once, the same way, rather than per engine.
`revision` (pin an HF revision) is common to every engine on both sides and is
listed once per side rather than repeated in each table below.

| Common option | Env var | Engine default | Meaning |
|---|---|---|---|
| `revision` | `ENHANCE_REVISION` | (adapter's pinned revision) | HF Hub revision for the `ENHANCE_ENGINE` weights. |
| `revision` | `DENOISE_REVISION` | (adapter's pinned revision) | HF Hub revision for the `DENOISE_ENGINE` weights. |

### `ENHANCE_ENGINE` (`load_sr`) options

| Engine | Input rate | Output rate | License |
|---|---|---|---|
| `callenhancer` (default) | 16 kHz | 48 kHz | CC-BY-NC-4.0 |
| `sidon` | 16 kHz | 48 kHz | MIT |
| `novasr` | 16 kHz | 48 kHz | Apache-2.0 |
| `lavasr` | flexible (8-48 kHz) | 48 kHz | Apache-2.0 |
| `apbwe` | flexible (bandlimited to 12 kHz) | 48 kHz | MIT |
| `hifiganbwe` | flexible | 48 kHz | MIT |
| `flowhigh` | flexible | 48 kHz | MIT |

`apbwe`, `hifiganbwe` and `novasr` take no options beyond `revision`.

**callenhancer**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `precision` | `ENHANCE_PRECISION` | `fp32` | Feature-extractor weight precision: `fp32` (full fidelity, ~2.3 GB) or `int8` (~580 MB, ~4x smaller and faster but audibly lossy on this model; see audiosronnx's own README for measured figures). |
| `chunk_seconds` | `ENHANCE_CHUNK_SECONDS` | `0` | `0` runs one straight, length-invariant pass over the whole clip (recommended). A positive value windows the audio with a 2 s crossfaded overlap, trading a little seam overhead for lower peak memory on very long calls. |
| `fe_path` | `ENHANCE_FE_PATH` | (none) | Local feature-extractor ONNX file, overriding the Hub download and `precision`. |
| `decoder_path` | `ENHANCE_DECODER_PATH` | (none) | Local decoder ONNX file, overriding the Hub download. |

**sidon**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `chunk_samples` | `ENHANCE_CHUNK_SAMPLES` | `1536000` (96 s at 16 kHz) | Window fed to the feature extractor per step. The w2v-BERT attention is O(T²), so a smaller value trades a little seam overhead for a much lower peak footprint. |
| `fe_path` | `ENHANCE_FE_PATH` | (none) | Local feature-extractor ONNX file, overriding the Hub download. |
| `decoder_path` | `ENHANCE_DECODER_PATH` | (none) | Local decoder ONNX file, overriding the Hub download. |

**lavasr**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `denoise` | `ENHANCE_DENOISE` | `false` | Run LavaSR's own optional UL-UNAS denoiser before bandwidth extension. Distinct from `DENOISE_ENGINE`, which runs as a separate stage before `ENHANCE_ENGINE` regardless of this flag. |
| `cutoff_hz` | `ENHANCE_CUTOFF_HZ` | `min(sample_rate, 16000) / 2` | Frequency below which the original signal is preserved in the spectral merge. |

**flowhigh**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `steps` | `ENHANCE_STEPS` | `4` | Euler steps for the flow-matching ODE. More steps cost proportionally more compute. |
| `seed` | `ENHANCE_SEED` | `0` | Seed for the noise the sampler starts from. |
| `merge` | `ENHANCE_MERGE` | `true` | Keep the input's own low band and take only the reconstructed high band. Disabling it returns the vocoder output unmodified. |
| `flow_path` | `ENHANCE_FLOW_PATH` | (none) | Local flow-matching ONNX file, overriding the Hub download. |
| `vocoder_path` | `ENHANCE_VOCODER_PATH` | (none) | Local BigVGAN vocoder ONNX file, overriding the Hub download. |

### `DENOISE_ENGINE` (`load_denoise`) options

| Engine | Native rate | Kind | License |
|---|---|---|---|
| `cmgan` | 16 kHz | denoise | MIT |
| `deepfilternet` | 48 kHz | denoise | MIT |
| `dpdfnet` | 8/16/48 kHz (by `model`) | denoise | Apache-2.0 |
| `frcrn` | 16 kHz | denoise | Apache-2.0 |
| `gtcrn` | 16 kHz | denoise | MIT |
| `metadenoiser` | 16 kHz | denoise | CC-BY-NC-4.0 |
| `mossformer2` | 48 kHz | denoise | Apache-2.0 |
| `mossformergan` | 16 kHz | denoise | Apache-2.0 |
| `mpsenet` | 16 kHz | denoise | MIT |
| `voicefixer` | 44.1 kHz | enhance (holistic restoration; loadable via `load_denoise`) | MIT |

`cmgan`, `frcrn`, `gtcrn` and `mossformergan` take no options beyond `revision`
and `onnx_path`.

**onnx_path** (`DENOISE_ONNX_PATH`, default none): a local ONNX file overriding
the Hub download. Accepted by `cmgan`, `frcrn`, `gtcrn`, `mossformergan`,
`dpdfnet`, `metadenoiser` and `mpsenet`.

**deepfilternet**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `mask_only` | `DENOISE_MASK_ONLY` | `false` | Skip the deep-filtering stage and return the ERB-masked signal: cheaper, and a useful ablation, but leaves detail the deep filter would recover. |
| `enc_path` | `DENOISE_ENC_PATH` | (none) | Local encoder ONNX file, overriding the Hub download. |
| `erb_dec_path` | `DENOISE_ERB_DEC_PATH` | (none) | Local ERB-decoder ONNX file, overriding the Hub download. |
| `df_dec_path` | `DENOISE_DF_DEC_PATH` | (none) | Local deep-filter-decoder ONNX file, overriding the Hub download. |

**dpdfnet**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `model` | `DENOISE_MODEL` | `dpdfnet2_48khz` | Published variant to run: `baseline`, `dpdfnet2`, `dpdfnet4`, `dpdfnet8` (16 kHz), `dpdfnet2_8khz`, `dpdfnet8_8khz` (8 kHz), `dpdfnet2_48khz`, `dpdfnet8_48khz` (48 kHz). Determines the native sample rate the model runs at. |
| `attn_limit_db` | `DENOISE_ATTN_LIMIT_DB` | `None` (unlimited) | Caps the maximum attenuation, in dB, by blending the noisy signal back in. A finite value such as `12` keeps a floor of residual noise, which sounds more natural than aggressive gating. |

**metadenoiser**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `model` | `DENOISE_MODEL` | `dns64` | `dns64` (33.5M params) or `dns48` (18.9M, roughly half the size). |

**mossformer2**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `dither` | `DENOISE_DITHER` | `0.0` | Kaldi filterbank dither. `0` keeps the engine deterministic; upstream uses `1.0`, which changes the output by about 1e-5. |

**mpsenet**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `model` | `DENOISE_MODEL` | `dns` | `dns` is trained on the DNS Challenge data and handles broadband noise far better; `vb` is the VoiceBank+DEMAND checkpoint the published PESQ figures come from. |

**voicefixer**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `analysis_path` | `DENOISE_ANALYSIS_PATH` | (none) | Local ResUNet mel-predictor ONNX file, overriding the Hub download. |
| `vocoder_path` | `DENOISE_VOCODER_PATH` | (none) | Local TFGAN vocoder ONNX file, overriding the Hub download. |
| `mel_filters_path` | `DENOISE_MEL_FILTERS_PATH` | (none) | Local mel-filterbank file, overriding the Hub download. |


---
## 1 · Install

| Env var | |
|---|---|
| `ONNX_RUNTIME` | `cpu` installs `onnxruntime`; `cuda` installs `onnxruntime-gpu`; `rocm` prints AMD's own install instructions and nothing is installed here |


In [ ]:
onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()

%pip install --quiet --pre "audiosronnx>=0.9.0a2" "soundfile>=0.14.0" "tqdm>=4.70.1"

if onnx_runtime == "cuda":
    %pip install --quiet --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0"


---
## 2 · Load Manifest

Reads the input manifest. Audio paths in it are relative to the manifest's
own directory, and every row must already carry `audio` (this notebook does
not synthesise or ingest speech itself) -- except with `SMOKE=True`, when a
manifest missing at `ENH_MANIFEST` is built from a few short edge-tts clips,
so the smoke path has something of its own to enhance.


In [ ]:
import json
from pathlib import Path


def _load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _output_manifest_path(manifest_path):
    """`<name>.enhanced.jsonl` beside the input (docs/manifest.md)."""
    p = Path(manifest_path)
    if p.suffix == ".jsonl":
        return p.with_name(p.stem + ".enhanced.jsonl")
    return p.with_name(p.name + ".enhanced.jsonl")


manifest_path = Path(_env("ENH_MANIFEST"))
if not manifest_path.is_file():
    if not SMOKE:
        raise RuntimeError(
            f"ENH_MANIFEST={manifest_path} does not name a file. Set it to the manifest this "
            f"notebook should enhance (docs/manifest.md), or set SMOKE=True for the built-in demo."
        )
    # SMOKE's built-in demo: this notebook has no default manifest of its own
    # (it only enhances audio another step already produced), so the smoke
    # path builds a couple of short edge-tts clips to enhance.
    %pip install --quiet --pre "edge-tts>=7.2.8" "nest-asyncio>=1.6.0" "soundfile>=0.14.0"
    import asyncio
    import subprocess

    import nest_asyncio
    nest_asyncio.apply()
    import edge_tts

    demo_dir = Path(_default_base_dir("tts_enhance_demo"))
    demo_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = demo_dir / "manifest.jsonl"
    DEMO_SENTENCES = [
        "Hey Mycroft, what is the weather like today in Lisbon?",
        "Set a timer for ten minutes.",
        "Play some jazz music in the kitchen.",
    ][:max(1, min(int(N_SAMPLES), 3))]
    DEMO_VOICE = "en-US-JennyNeural"

    async def _demo_synth(text, out_mp3):
        await edge_tts.Communicate(text, DEMO_VOICE).save(str(out_mp3))

    demo_rows = []
    for i, text in enumerate(DEMO_SENTENCES, 1):
        clip_id = f"demo_{i:04d}"
        mp3_path = demo_dir / f"{clip_id}.mp3"
        wav_path = demo_dir / f"{clip_id}.wav"
        asyncio.run(_demo_synth(text, mp3_path))
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(mp3_path), "-ar", "16000", "-ac", "1", str(wav_path)],
            check=True,
        )
        mp3_path.unlink()
        import soundfile as _sf
        info = _sf.info(str(wav_path))
        demo_rows.append({
            "id": clip_id, "text": text, "lang": "en-US", "audio": wav_path.name,
            "duration_s": round(info.frames / info.samplerate, 3), "sample_rate": info.samplerate,
            "source": {"kind": "tts", "plugin": "edge-tts", "voice": DEMO_VOICE},
        })
    with open(manifest_path, "w", encoding="utf-8") as f:
        for row in demo_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"SMOKE=True and ENH_MANIFEST does not exist: wrote a {len(demo_rows)}-row demo manifest to {manifest_path}")

manifest_dir = manifest_path.parent

input_rows = _load_jsonl(str(manifest_path))
if not input_rows:
    raise RuntimeError(f"{manifest_path} has no rows to enhance.")
if SMOKE:
    input_rows = input_rows[:N_SAMPLES]
    print(f"SMOKE=True: capped to {len(input_rows)} row(s).")

for row in input_rows:
    if not row.get("id"):
        raise RuntimeError("a row is missing its id field")
    if not row.get("audio"):
        raise RuntimeError(f"row {row['id']!r} has no audio field; every row this notebook "
                            f"reads must already carry audio (docs/pipeline.md, steps 2a/2b)")

output_path = _output_manifest_path(manifest_path)
audio_out_dir = manifest_dir / _env("ENH_AUDIO_DIR", "enhanced")

print(f"Manifest: {len(input_rows)} row(s) in {manifest_path}")
print(f"Output manifest will be {output_path}")
print(f"Enhanced audio will be written under {audio_out_dir}")


---
## 3 · Load the Enhancer and Denoiser

`audiosronnx.load_sr(ENHANCE_ENGINE, providers=..., cache_dir=..., **kwargs)`
and `audiosronnx.load_denoise(DENOISE_ENGINE, providers=..., cache_dir=...,
**kwargs)`. At least one of `ENHANCE_ENGINE` / `DENOISE_ENGINE` must be set;
this cell refuses to run with neither. When both are set, `DENOISE_ENGINE`
runs first at Stage 4 and `ENHANCE_ENGINE` sees its output.

Every engine adapter in `audiosronnx` accepts a `providers` list and a
`cache_dir` directly, so both are always passed through from `ONNX_PROVIDERS`
and `ENH_CACHE_DIR`. This cell reads each selected engine's own constructor
signature to decide which of its `ENHANCE_<OPTION>` / `DENOISE_<OPTION>`
values apply, and refuses one an engine does not accept, naming the engine and
the option (see the option tables above).


In [ ]:
import inspect

from audiosronnx import load_sr, load_denoise
from audiosronnx.base import get_engine

PROVIDER_PRIORITY = [
    "CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider", "CPUExecutionProvider",
]

# option name -> typed-parsing kind; the env var is <PREFIX>_<OPTION UPPERCASE>
ENHANCE_OPTION_TYPES = {
    "revision":       "str",
    "fe_path":        "str",
    "decoder_path":   "str",
    "chunk_seconds":  "float",
    "precision":      "str",
    "steps":          "int",
    "seed":           "int",
    "merge":          "bool",
    "flow_path":      "str",
    "vocoder_path":   "str",
    "denoise":        "bool",
    "cutoff_hz":      "float",
    "chunk_samples":  "int",
}

DENOISE_OPTION_TYPES = {
    "revision":          "str",
    "onnx_path":         "str",
    "mask_only":         "bool",
    "enc_path":          "str",
    "erb_dec_path":      "str",
    "df_dec_path":       "str",
    "model":             "str",
    "attn_limit_db":     "float",
    "dither":            "float",
    "analysis_path":     "str",
    "vocoder_path":      "str",
    "mel_filters_path":  "str",
}


def _parse_typed_env(env_name, raw, kind):
    if kind == "bool":
        low = raw.strip().lower()
        if low in ("1", "true", "yes", "on"):
            return True
        if low in ("0", "false", "no", "off"):
            return False
        raise ValueError(f"{env_name}={raw!r} is not a valid bool (use true/false)")
    if kind == "int":
        try:
            return int(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid int") from None
    if kind == "float":
        try:
            return float(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid float") from None
    if kind == "json":
        try:
            return json.loads(raw)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{env_name}={raw!r} is not valid JSON: {exc}") from None
    return raw


def _resolve_onnx_providers(raw):
    """A comma list, or "auto": CUDA, ROCm, MIGraphX, CPU, CPU always last (docs/pipeline.md)."""
    raw = (raw or "").strip()
    if not raw or raw.lower() == "auto":
        import onnxruntime as ort

        available = ort.get_available_providers()
        resolved = [p for p in PROVIDER_PRIORITY if p in available]
        if "CPUExecutionProvider" not in resolved:
            resolved.append("CPUExecutionProvider")
        return resolved
    return [p.strip() for p in raw.split(",") if p.strip()]


def _accepted_params(adapter_cls):
    """Constructor parameter names of the adapter, minus self and **cfg."""
    try:
        sig = inspect.signature(adapter_cls.__init__)
    except (TypeError, ValueError):
        return set()
    return {
        name for name, param in sig.parameters.items()
        if name != "self" and param.kind != inspect.Parameter.VAR_KEYWORD
    }


def _resolve_option_kwargs(prefix, option_types, adapter_cls, engine_alias):
    """Build the load_sr()/load_denoise() extra kwargs for `engine_alias` from every
    <prefix>_<OPTION> env var that is set, refusing one the adapter does not accept."""
    accepted = _accepted_params(adapter_cls)
    kwargs = {}
    for option, kind in option_types.items():
        env_name = f"{prefix}_{option.upper()}"
        raw = os.environ.get(env_name, "").strip()
        if not raw:
            continue
        if option not in accepted:
            raise RuntimeError(
                f"{env_name} is set but engine {engine_alias!r} does not accept an option "
                f"named {option!r}. Its constructor accepts: {sorted(accepted) or '(none)'}."
            )
        kwargs[option] = _parse_typed_env(env_name, raw, kind)
    return kwargs


enhance_engine = _env("ENHANCE_ENGINE", "callenhancer").strip()
denoise_engine = _env("DENOISE_ENGINE", "").strip()
if not enhance_engine and not denoise_engine:
    raise RuntimeError(
        "Set ENHANCE_ENGINE, DENOISE_ENGINE, or both: at least one stage must run to "
        "produce output."
    )

onnx_providers = _resolve_onnx_providers(_env("ONNX_PROVIDERS", "auto"))
cache_dir = _env("ENH_CACHE_DIR").strip() or None

denoiser = None
denoise_kwargs = {}
if denoise_engine:
    denoise_entry = get_engine(denoise_engine)
    denoise_kwargs = _resolve_option_kwargs(
        "DENOISE", DENOISE_OPTION_TYPES, denoise_entry.adapter_class, denoise_engine
    )
    denoiser = load_denoise(denoise_engine, providers=onnx_providers, cache_dir=cache_dir, **denoise_kwargs)
    print(f"Denoiser ready: engine={denoise_engine}  native_rate={denoiser.input_sample_rate} Hz  "
          f"options={denoise_kwargs or '(engine defaults)'}")

enhancer = None
enhance_kwargs = {}
if enhance_engine:
    enhance_entry = get_engine(enhance_engine)
    enhance_kwargs = _resolve_option_kwargs(
        "ENHANCE", ENHANCE_OPTION_TYPES, enhance_entry.adapter_class, enhance_engine
    )
    enhancer = load_sr(enhance_engine, providers=onnx_providers, cache_dir=cache_dir, **enhance_kwargs)
    engine_input_rate = enhancer.input_sample_rate or None  # None = the engine accepts any input rate
    print(f"Enhancer ready: engine={enhance_engine}  input_rate={engine_input_rate or 'flexible'} Hz  "
          f"output_rate={enhancer.output_sample_rate} Hz  options={enhance_kwargs or '(engine defaults)'}")

print(f"Providers: {onnx_providers}")
if cache_dir:
    print(f"Model cache: {cache_dir}")

---
## 4 · Enhance Every Row

Writes one new row per input row, following [manifest.md](../docs/manifest.md):
`id` is `<parent id>-enh`, `parent_id` names the source row, and `audio`,
`duration_s` and `sample_rate` describe the enhanced file. `text`, `lang`,
`source` and `speaker` are copied from the parent row unchanged. Neither this
notebook nor `audiosronnx` deletes or moves the source audio.

When `DENOISE_ENGINE` is set, its `denoise()` runs first, at its own native
sample rate; when `ENHANCE_ENGINE` is also set, `upscale()` then runs on the
denoised array (not the source file), at the rate the denoiser produced. When
only `ENHANCE_ENGINE` is set, `upscale()` runs directly on the source file.
The final array is resampled once, if needed, from whichever stage ran last to
`TARGET_SR`. Every row's new `provenance` field, and the `.meta.json` sidecar,
record which engine(s) ran and every option actually passed to each.

A row whose output WAV already exists with a nonzero size is left alone
(resume-safe) unless `ENH_FORCE=1`; the output manifest is always rewritten in
full, so its row count matches the input row count on every run, whether a
row was freshly processed or skipped. Writing zero rows is a failure, never a
silently empty output.

In [ ]:
import hashlib
import sys
import time

import soundfile as sf
from tqdm import tqdm

from audiosronnx.audio import resample as _audiosr_resample
from audiosronnx.audio import write_wav as _audiosr_write_wav

target_sr = int(_env("TARGET_SR", "24000"))
force = _flag("ENH_FORCE")
audio_dir_name = _env("ENH_AUDIO_DIR", "enhanced")


def _sha256_file(path, block_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block_size), b""):
            h.update(chunk)
    return h.hexdigest()


def _enhance_fingerprint(src_path, denoise_engine, denoise_kwargs, enhance_engine, enhance_kwargs, target_sr):
    """The configuration that made this output: the source audio's own bytes,
    plus every engine and option that touched them. Two runs that ever
    disagree on any of these must not share an output WAV."""
    payload = {
        "source_sha256": _sha256_file(src_path),
        "denoise_engine": denoise_engine,
        "denoise_options": denoise_kwargs,
        "enhance_engine": enhance_engine,
        "enhance_options": enhance_kwargs,
        "target_sr": target_sr,
    }
    blob = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest(), payload


def enhance_row(row, manifest_dir, audio_out_dir, denoiser, enhancer, denoise_engine, enhance_engine,
                denoise_kwargs, enhance_kwargs, target_sr, force):
    """Denoise (if configured) then enhance (if configured) one row's audio.

    Returns (new_row, was_reused); new_row carries the provenance of both stages."""
    src_path = manifest_dir / row["audio"]
    if not src_path.is_file():
        raise RuntimeError(f"row {row['id']!r}: audio file not found at {src_path}")

    out_id = f"{row['id']}-enh"
    out_rel = f"{audio_dir_name}/{out_id}.wav"
    out_abs = manifest_dir / out_rel
    fp_abs = manifest_dir / f"{audio_dir_name}/{out_id}.fp.json"

    provenance = {
        "denoise": {"engine": denoise_engine, "options": denoise_kwargs} if denoiser is not None else None,
        "enhance": {"engine": enhance_engine, "options": enhance_kwargs} if enhancer is not None else None,
    }

    fingerprint, fingerprint_payload = _enhance_fingerprint(
        src_path, denoise_engine, denoise_kwargs, enhance_engine, enhance_kwargs, target_sr
    )

    reused = False
    if out_abs.exists() and out_abs.stat().st_size > 0 and fp_abs.is_file() and not force:
        try:
            prior = json.loads(fp_abs.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            prior = None
        if prior and prior.get("fingerprint") == fingerprint:
            reused = True

    if reused:
        skipped = True
    else:
        skipped = False
        out_abs.parent.mkdir(parents=True, exist_ok=True)

        native_rate = sf.info(str(src_path)).samplerate
        print(f"  [{row['id']}] input {native_rate} Hz")

        audio_arr = str(src_path)  # a str is still-unprocessed: the source file path
        rate = None
        if denoiser is not None:
            audio_arr, rate = denoiser.denoise(audio_arr)
            print(f"  [{row['id']}] denoise({denoise_engine}) -> {rate} Hz")
        if enhancer is not None:
            if isinstance(audio_arr, str):
                audio_arr, rate = enhancer.upscale(audio_arr)
            else:
                audio_arr, rate = enhancer.upscale(audio_arr, sample_rate=rate)
            print(f"  [{row['id']}] enhance({enhance_engine}) -> {rate} Hz")

        if isinstance(audio_arr, str):
            # neither stage ran; the Stage 3 guard makes this unreachable.
            raise RuntimeError("no denoise or enhance stage ran")

        if rate != target_sr:
            audio_arr = _audiosr_resample(audio_arr, rate, target_sr)
            rate = target_sr
        _audiosr_write_wav(str(out_abs), audio_arr, rate)
        fp_abs.write_text(
            json.dumps({"fingerprint": fingerprint, **fingerprint_payload}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    info = sf.info(str(out_abs))
    new_row = {
        "id": out_id,
        "parent_id": row["id"],
        "audio": out_rel,
        "duration_s": round(info.duration, 3),
        "sample_rate": info.samplerate,
        "text": row["text"],
        "lang": row["lang"],
        "source": row["source"],
        "speaker": row.get("speaker"),
        "provenance": provenance,
    }
    return new_row, skipped


output_rows = []
n_enhanced, n_skipped = 0, 0
start = time.time()
for row in tqdm(input_rows, desc="Enhance", unit="row", file=sys.stdout):
    new_row, was_skipped = enhance_row(
        row, manifest_dir, audio_out_dir, denoiser, enhancer, denoise_engine, enhance_engine,
        denoise_kwargs, enhance_kwargs, target_sr, force
    )
    output_rows.append(new_row)
    if was_skipped:
        n_skipped += 1
    else:
        n_enhanced += 1
elapsed = time.time() - start

if len(output_rows) == 0:
    raise RuntimeError("zero rows written: nothing was enhanced")

# A real join, not a count: every input row's own id must appear exactly once
# as a parent_id among the outputs. A count that only matched in length would
# stay green even if the loop above silently dropped one row and duplicated
# another.
_input_ids = {row["id"] for row in input_rows}
_output_parent_ids = [r["parent_id"] for r in output_rows]
assert len(_output_parent_ids) == len(set(_output_parent_ids)), (
    "a row was enhanced more than once"
)
assert set(_output_parent_ids) == _input_ids, (
    f"output rows do not match input rows 1:1: "
    f"missing={_input_ids - set(_output_parent_ids)}  "
    f"unexpected={set(_output_parent_ids) - _input_ids}"
)

with open(output_path, "w", encoding="utf-8") as f:
    for r in output_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

from importlib.metadata import version as _pkg_version

meta = {
    "notebooks": ["tts_enhance"],
    "manifest": manifest_path.name,
    "output": output_path.name,
    "rows_total": len(output_rows),
    "rows_enhanced_this_run": n_enhanced,
    "rows_already_enhanced": n_skipped,
    "instruments": {
        "audio": {
            "package": "audiosronnx",
            "version": _pkg_version("audiosronnx"),
            "target_sample_rate": target_sr,
            "onnx_providers": onnx_providers,
            "denoise": (
                {
                    "engine": denoise_engine,
                    "options": denoise_kwargs,
                    "native_sample_rate": denoiser.input_sample_rate,
                }
                if denoiser is not None else None
            ),
            "enhance": (
                {
                    "engine": enhance_engine,
                    "options": enhance_kwargs,
                    "model": get_engine(enhance_engine).description,
                    "license": get_engine(enhance_engine).license,
                    "input_sample_rate": get_engine(enhance_engine).input_sample_rate or "flexible",
                    "output_sample_rate": get_engine(enhance_engine).output_sample_rate,
                }
                if enhancer is not None else None
            ),
        },
    },
}
with open(str(output_path) + ".meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"\n{n_enhanced} enhanced, {n_skipped} reused (matching fingerprint), "
      f"{len(output_rows)} row(s) -> {output_path} in {elapsed:.1f}s")
print(f"Sidecar: {output_path}.meta.json")


---
## 5 · Output Summary

Confirms the output manifest holds exactly one row per input row, and that
every row's enhanced audio exists on disk.


In [ ]:
print("=" * 65)
print("  OUTPUT SUMMARY")
print("=" * 65)
_input_ids = {row["id"] for row in input_rows}
_output_parent_ids = [r["parent_id"] for r in output_rows]
assert len(_output_parent_ids) == len(set(_output_parent_ids)), (
    "a row was enhanced more than once"
)
assert set(_output_parent_ids) == _input_ids, (
    f"output rows do not match input rows 1:1: "
    f"missing={_input_ids - set(_output_parent_ids)}  "
    f"unexpected={set(_output_parent_ids) - _input_ids}"
)
for r in output_rows:
    audio_path = manifest_dir / r["audio"]
    assert audio_path.is_file() and audio_path.stat().st_size > 0, (
        f"{r['id']}: missing or empty audio at {audio_path}"
    )
print(f"  input rows      : {len(input_rows)}")
print(f"  output rows     : {len(output_rows)}  ->  {output_path}")
print(f"  enhanced audio  : {audio_out_dir}")
print("=" * 65)


---
Licensed under Apache-2.0.
